In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2
import zlib
import base64

In [2]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

In [3]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

In [4]:
start_date = datetime(2020, 1, 1)
end_date = datetime(2024, 5, 1)

current_date = start_date

In [5]:
while current_date < end_date:
	next_month = current_date + timedelta(days=31)
	next_month = next_month.replace(day=1)

	# Formatear las fechas para la consulta SQL
	start_date_str = current_date.strftime('%d/%m/%Y')
	end_date_str = next_month.strftime('%d/%m/%Y')

	query = f"""SELECT
		RAS.REDASISCOD,
		RAS.REDASISDES,
		CAS.CENASICOD,
		CAS.CENASIDES,
		DIA.DIAGCOD,
		DIA.DIAGDES,
		ACE.ATENAMBDIAGORD,
		--B.SOLEXAFEC,
		E.SOLEXDFECCITA,
		PA.PERTIPDOCIDENCOD AS TIPDOCMED,
		PA.PERDOCIDENNUM AS NUMDOCMED,
		PA.PERNACFEC AS MEDNACFEC,
		B.SOLEXAAREHOSCOD,
		AHO.AREHOSDES,
		B.SOLEXASERVHOSCOD,
		SHO.SERVHOSDES,
		B.SOLEXAACTCOD,
		ACT.ACTDES,
		G.PERNACFEC,
		G.PERTIPDOCIDENCOD,
		G.PERDOCIDENNUM,
		G.PERSEXOCOD,
		A.RESEXAFEC,
		A.RESEXACPSCOD,
		CPP.CPSDES,
		D.RESEXDEXADES,
		D.RESEXDEXAUND,
		D.RESEXDEXANORVAL
	FROM CTDAA10 ACE
	LEFT OUTER JOIN 
		ETSEA10 B ON 
			ACE.ATENAMBORICENASICOD  = B.SOLEXAORICENASIORICOD
				AND ACE.ATENAMBCENASICOD  = B.SOLEXACENASIORICOD
				AND ACE.ATENAMBNUM  = B.SOLEXAACTMEDORINUM
	LEFT OUTER JOIN
		ETSED10 E ON
			E.SOLEXAORICENASICOD = B.SOLEXAORICENASICOD
				AND E.SOLEXACENASICOD = B.SOLEXACENASICOD
				AND E.SOLEXATIPEXACOD = B.SOLEXATIPEXACOD 
				AND E.SOLEXANUM = B.SOLEXANUM
	LEFT OUTER JOIN 
		ETREA10 A ON 
			E.SOLEXAORICENASICOD = A.RESEXAORICENASICOD
				AND E.SOLEXACENASICOD = A.RESEXACENASICOD
				AND E.SOLEXATIPEXACOD = A.RESEXATIPEXACOD
				AND E.SOLEXANUM = A.RESEXASOLEXANUM
				AND E.SOLEXDCPSCOD = A.RESEXACPSCOD
	LEFT OUTER JOIN 
		ETRED10 D ON 
			D.RESEXAORICENASICOD = A.RESEXAORICENASICOD
				AND D.RESEXACENASICOD = A.RESEXACENASICOD
				AND D.RESEXATIPEXACOD = A.RESEXATIPEXACOD
				AND D.RESEXASOLEXANUM = A.RESEXASOLEXANUM
				AND D.RESEXACPSCOD = A.RESEXACPSCOD
	LEFT OUTER JOIN 
		CMPER10 G ON 
			G.PERSECNUM = B.SOLEXAPACSECNUM
	LEFT OUTER JOIN
		CMPRS10 PR ON
			B.SOLEXATIPDOCIDENPERCOD = PR.TIPDOCIDENPERCOD 
				AND B.SOLEXAPERASISDOCIDENNUM = PR.PERASISDOCIDENNUM
	LEFT OUTER JOIN
		CMPER10 PA ON
			B.SOLEXATIPDOCIDENPERCOD = PA.PERTIPDOCIDENCOD 
				AND B.SOLEXAPERASISDOCIDENNUM = PA.PERDOCIDENNUM
	LEFT OUTER JOIN
		CMCAS10 CAS ON
			ACE.ATENAMBCENASICOD = CAS.CENASICOD
	LEFT OUTER JOIN
		CMRAS10 RAS ON
			CAS.REDASISCOD = RAS.REDASISCOD
	LEFT OUTER JOIN
		CMDIA10 DIA ON
			ACE.DIAGCOD = DIA.DIAGCOD
	LEFT OUTER JOIN
		CMAHO10 AHO ON
			B.SOLEXAAREHOSCOD = AHO.AREHOSCOD
	LEFT OUTER JOIN
		CMSHO10 SHO ON
			B.SOLEXASERVHOSCOD = SHO.SERVHOSCOD
	LEFT OUTER JOIN
		CMACT10 ACT ON
			B.SOLEXAACTCOD = ACT.ACTCOD
	LEFT OUTER JOIN
		CMCPP10 CPP ON
			A.RESEXACPSCOD = CPP.CPSCOD
	WHERE 	
		B.SOLEXAFEC >= TO_DATE('{start_date_str}', 'DD/MM/YYYY') AND B.SOLEXAFEC < TO_DATE('{end_date_str}', 'DD/MM/YYYY')
		AND PA.PERESTREGCOD = '1'
		AND PR.PERASISESTREGCOD = '1'
		AND A.RESEXACPSCOD IN ('82465', '82947')
		AND ACE.DIAGCOD LIKE '%E1%'
	"""
	print(f'PROCESANDO DE: {start_date_str} A {end_date_str}')
	base = pd.read_sql_query(query, con=connection0)

	# Agrupar por 'solexdfeccita' y 'perdocidennum'
	grouped = base.groupby(['solexdfeccita', 'perdocidennum'])
	new_rows = []
	for name, group in grouped:
		row = {}
		row.update({col: group.iloc[0][col] for col in base.columns[:-6]})
		if len(group) > 1:
			row.update({f"{col}1": group.iloc[0][col] for col in base.columns[-6:]})
			row.update({f"{col}2": group.iloc[1][col] for col in base.columns[-6:]})
		else:
			row.update({f"{col}1": group.iloc[0][col] for col in base.columns[-6:]})
			row.update({f"{col}2": None for col in base.columns[-6:]})
		new_rows.append(row)

	# Crear un nuevo DataFrame con los resultados
	df_combined = pd.DataFrame(new_rows)

	# Guardar los resultados en la base de datos
	df_combined.to_sql(name=f'pnda_labo_cext_diabetes', con=engine1, if_exists='append', index=False)

	# Mover al próximo mes
	current_date = next_month


PROCESANDO DE: 01/01/2020 A 01/02/2020
PROCESANDO DE: 01/02/2020 A 01/03/2020
PROCESANDO DE: 01/03/2020 A 01/04/2020
PROCESANDO DE: 01/04/2020 A 01/05/2020
PROCESANDO DE: 01/05/2020 A 01/06/2020
PROCESANDO DE: 01/06/2020 A 01/07/2020
PROCESANDO DE: 01/07/2020 A 01/08/2020
PROCESANDO DE: 01/08/2020 A 01/09/2020
PROCESANDO DE: 01/09/2020 A 01/10/2020
PROCESANDO DE: 01/10/2020 A 01/11/2020
PROCESANDO DE: 01/11/2020 A 01/12/2020
PROCESANDO DE: 01/12/2020 A 01/01/2021
PROCESANDO DE: 01/01/2021 A 01/02/2021
PROCESANDO DE: 01/02/2021 A 01/03/2021
PROCESANDO DE: 01/03/2021 A 01/04/2021
PROCESANDO DE: 01/04/2021 A 01/05/2021
PROCESANDO DE: 01/05/2021 A 01/06/2021
PROCESANDO DE: 01/06/2021 A 01/07/2021
PROCESANDO DE: 01/07/2021 A 01/08/2021
PROCESANDO DE: 01/08/2021 A 01/09/2021
PROCESANDO DE: 01/09/2021 A 01/10/2021
PROCESANDO DE: 01/10/2021 A 01/11/2021
PROCESANDO DE: 01/11/2021 A 01/12/2021
PROCESANDO DE: 01/12/2021 A 01/01/2022
PROCESANDO DE: 01/01/2022 A 01/02/2022
PROCESANDO DE: 01/02/2022

In [6]:
start_date = datetime(2020, 1, 1)
end_date = datetime(2024, 5, 1)

current_date = start_date

In [7]:
while current_date < end_date:
	next_month = current_date + timedelta(days=31)
	next_month = next_month.replace(day=1)

	# Formatear las fechas para la consulta SQL
	start_date_str = current_date.strftime('%d/%m/%Y')
	end_date_str = next_month.strftime('%d/%m/%Y')

	query = f"""SELECT
		RAS.REDASISCOD,
		RAS.REDASISDES,
		CAS.CENASICOD,
		CAS.CENASIDES,
		DIA.DIAGCOD,
		DIA.DIAGDES,
		ACE.ATENAMBDIAGORD,
		--B.SOLEXAFEC,
		E.SOLEXDFECCITA,
		PA.PERTIPDOCIDENCOD AS TIPDOCMED,
		PA.PERDOCIDENNUM AS NUMDOCMED,
		PA.PERNACFEC AS MEDNACFEC,
		B.SOLEXAAREHOSCOD,
		AHO.AREHOSDES,
		B.SOLEXASERVHOSCOD,
		SHO.SERVHOSDES,
		B.SOLEXAACTCOD,
		ACT.ACTDES,
		G.PERNACFEC,
		G.PERTIPDOCIDENCOD,
		G.PERDOCIDENNUM,
		G.PERSEXOCOD,
		A.RESEXAFEC,
		A.RESEXACPSCOD,
		CPP.CPSDES,
		D.RESEXDEXADES,
		D.RESEXDEXAUND,
		D.RESEXDEXANORVAL
	FROM CTDAA10 ACE
	LEFT OUTER JOIN 
		ETSEA10 B ON 
			ACE.ATENAMBORICENASICOD  = B.SOLEXAORICENASIORICOD
				AND ACE.ATENAMBCENASICOD  = B.SOLEXACENASIORICOD
				AND ACE.ATENAMBNUM  = B.SOLEXAACTMEDORINUM
	LEFT OUTER JOIN
		ETSED10 E ON
			E.SOLEXAORICENASICOD = B.SOLEXAORICENASICOD
				AND E.SOLEXACENASICOD = B.SOLEXACENASICOD
				AND E.SOLEXATIPEXACOD = B.SOLEXATIPEXACOD 
				AND E.SOLEXANUM = B.SOLEXANUM
	LEFT OUTER JOIN 
		ETREA10 A ON 
			E.SOLEXAORICENASICOD = A.RESEXAORICENASICOD
				AND E.SOLEXACENASICOD = A.RESEXACENASICOD
				AND E.SOLEXATIPEXACOD = A.RESEXATIPEXACOD
				AND E.SOLEXANUM = A.RESEXASOLEXANUM
				AND E.SOLEXDCPSCOD = A.RESEXACPSCOD
	LEFT OUTER JOIN 
		ETRED10 D ON 
			D.RESEXAORICENASICOD = A.RESEXAORICENASICOD
				AND D.RESEXACENASICOD = A.RESEXACENASICOD
				AND D.RESEXATIPEXACOD = A.RESEXATIPEXACOD
				AND D.RESEXASOLEXANUM = A.RESEXASOLEXANUM
				AND D.RESEXACPSCOD = A.RESEXACPSCOD
	LEFT OUTER JOIN 
		CMPER10 G ON 
			G.PERSECNUM = B.SOLEXAPACSECNUM
	LEFT OUTER JOIN
		CMPRS10 PR ON
			B.SOLEXATIPDOCIDENPERCOD = PR.TIPDOCIDENPERCOD 
				AND B.SOLEXAPERASISDOCIDENNUM = PR.PERASISDOCIDENNUM
	LEFT OUTER JOIN
		CMPER10 PA ON
			B.SOLEXATIPDOCIDENPERCOD = PA.PERTIPDOCIDENCOD 
				AND B.SOLEXAPERASISDOCIDENNUM = PA.PERDOCIDENNUM
	LEFT OUTER JOIN
		CMCAS10 CAS ON
			ACE.ATENAMBCENASICOD = CAS.CENASICOD
	LEFT OUTER JOIN
		CMRAS10 RAS ON
			CAS.REDASISCOD = RAS.REDASISCOD
	LEFT OUTER JOIN
		CMDIA10 DIA ON
			ACE.DIAGCOD = DIA.DIAGCOD
	LEFT OUTER JOIN
		CMAHO10 AHO ON
			B.SOLEXAAREHOSCOD = AHO.AREHOSCOD
	LEFT OUTER JOIN
		CMSHO10 SHO ON
			B.SOLEXASERVHOSCOD = SHO.SERVHOSCOD
	LEFT OUTER JOIN
		CMACT10 ACT ON
			B.SOLEXAACTCOD = ACT.ACTCOD
	LEFT OUTER JOIN
		CMCPP10 CPP ON
			A.RESEXACPSCOD = CPP.CPSCOD
	WHERE 	
		B.SOLEXAFEC >= TO_DATE('{start_date_str}', 'DD/MM/YYYY') AND B.SOLEXAFEC < TO_DATE('{end_date_str}', 'DD/MM/YYYY')
		AND PA.PERESTREGCOD = '1'
		AND PR.PERASISESTREGCOD = '1'
		AND A.RESEXACPSCOD IN ('82947', '82565')
		AND (ACE.DIAGCOD LIKE '%N18%' OR ACE.DIAGCOD LIKE '%I12.%')
	"""
	print(f'PROCESANDO DE: {start_date_str} A {end_date_str}')
	base = pd.read_sql_query(query, con=connection0)

	# Agrupar por 'solexdfeccita' y 'perdocidennum'
	grouped = base.groupby(['solexdfeccita', 'perdocidennum'])
	new_rows = []
	for name, group in grouped:
		row = {}
		row.update({col: group.iloc[0][col] for col in base.columns[:-6]})
		if len(group) > 1:
			row.update({f"{col}1": group.iloc[0][col] for col in base.columns[-6:]})
			row.update({f"{col}2": group.iloc[1][col] for col in base.columns[-6:]})
		else:
			row.update({f"{col}1": group.iloc[0][col] for col in base.columns[-6:]})
			row.update({f"{col}2": None for col in base.columns[-6:]})
		new_rows.append(row)

	# Crear un nuevo DataFrame con los resultados
	df_combined = pd.DataFrame(new_rows)

	# Guardar los resultados en la base de datos
	df_combined.to_sql(name=f'pnda_labo_cext_enf_renal', con=engine1, if_exists='append', index=False)

	# Mover al próximo mes
	current_date = next_month

PROCESANDO DE: 01/01/2020 A 01/02/2020
PROCESANDO DE: 01/02/2020 A 01/03/2020
PROCESANDO DE: 01/03/2020 A 01/04/2020
PROCESANDO DE: 01/04/2020 A 01/05/2020
PROCESANDO DE: 01/05/2020 A 01/06/2020
PROCESANDO DE: 01/06/2020 A 01/07/2020
PROCESANDO DE: 01/07/2020 A 01/08/2020
PROCESANDO DE: 01/08/2020 A 01/09/2020
PROCESANDO DE: 01/09/2020 A 01/10/2020
PROCESANDO DE: 01/10/2020 A 01/11/2020
PROCESANDO DE: 01/11/2020 A 01/12/2020
PROCESANDO DE: 01/12/2020 A 01/01/2021
PROCESANDO DE: 01/01/2021 A 01/02/2021
PROCESANDO DE: 01/02/2021 A 01/03/2021
PROCESANDO DE: 01/03/2021 A 01/04/2021
PROCESANDO DE: 01/04/2021 A 01/05/2021
PROCESANDO DE: 01/05/2021 A 01/06/2021
PROCESANDO DE: 01/06/2021 A 01/07/2021
PROCESANDO DE: 01/07/2021 A 01/08/2021
PROCESANDO DE: 01/08/2021 A 01/09/2021
PROCESANDO DE: 01/09/2021 A 01/10/2021
PROCESANDO DE: 01/10/2021 A 01/11/2021
PROCESANDO DE: 01/11/2021 A 01/12/2021
PROCESANDO DE: 01/12/2021 A 01/01/2022
PROCESANDO DE: 01/01/2022 A 01/02/2022
PROCESANDO DE: 01/02/2022

In [8]:
start_date = datetime(2020, 1, 1)
end_date = datetime(2024, 5, 1)

current_date = start_date

In [9]:
while current_date < end_date:
	next_month = current_date + timedelta(days=31)
	next_month = next_month.replace(day=1)

	# Formatear las fechas para la consulta SQL
	start_date_str = current_date.strftime('%d/%m/%Y')
	end_date_str = next_month.strftime('%d/%m/%Y')

	query = f"""SELECT
		RAS.REDASISCOD,
		RAS.REDASISDES,
		CAS.CENASICOD,
		CAS.CENASIDES,
		DIA.DIAGCOD,
		DIA.DIAGDES,
		ACE.ATENAMBDIAGORD,
		--B.SOLEXAFEC,
		E.SOLEXDFECCITA,
		PA.PERTIPDOCIDENCOD AS TIPDOCMED,
		PA.PERDOCIDENNUM AS NUMDOCMED,
		PA.PERNACFEC AS MEDNACFEC,
		B.SOLEXAAREHOSCOD,
		AHO.AREHOSDES,
		B.SOLEXASERVHOSCOD,
		SHO.SERVHOSDES,
		B.SOLEXAACTCOD,
		ACT.ACTDES,
		G.PERNACFEC,
		G.PERTIPDOCIDENCOD,
		G.PERDOCIDENNUM,
		G.PERSEXOCOD,
		A.RESEXAFEC,
		A.RESEXACPSCOD,
		CPP.CPSDES,
		D.RESEXDEXADES,
		D.RESEXDEXAUND,
		D.RESEXDEXANORVAL
	FROM CTDAA10 ACE
	LEFT OUTER JOIN 
		ETSEA10 B ON 
			ACE.ATENAMBORICENASICOD  = B.SOLEXAORICENASIORICOD
				AND ACE.ATENAMBCENASICOD  = B.SOLEXACENASIORICOD
				AND ACE.ATENAMBNUM  = B.SOLEXAACTMEDORINUM
	LEFT OUTER JOIN
		ETSED10 E ON
			E.SOLEXAORICENASICOD = B.SOLEXAORICENASICOD
				AND E.SOLEXACENASICOD = B.SOLEXACENASICOD
				AND E.SOLEXATIPEXACOD = B.SOLEXATIPEXACOD 
				AND E.SOLEXANUM = B.SOLEXANUM
	LEFT OUTER JOIN 
		ETREA10 A ON 
			E.SOLEXAORICENASICOD = A.RESEXAORICENASICOD
				AND E.SOLEXACENASICOD = A.RESEXACENASICOD
				AND E.SOLEXATIPEXACOD = A.RESEXATIPEXACOD
				AND E.SOLEXANUM = A.RESEXASOLEXANUM
				AND E.SOLEXDCPSCOD = A.RESEXACPSCOD
	LEFT OUTER JOIN 
		ETRED10 D ON 
			D.RESEXAORICENASICOD = A.RESEXAORICENASICOD
				AND D.RESEXACENASICOD = A.RESEXACENASICOD
				AND D.RESEXATIPEXACOD = A.RESEXATIPEXACOD
				AND D.RESEXASOLEXANUM = A.RESEXASOLEXANUM
				AND D.RESEXACPSCOD = A.RESEXACPSCOD
	LEFT OUTER JOIN 
		CMPER10 G ON 
			G.PERSECNUM = B.SOLEXAPACSECNUM
	LEFT OUTER JOIN
		CMPRS10 PR ON
			B.SOLEXATIPDOCIDENPERCOD = PR.TIPDOCIDENPERCOD 
				AND B.SOLEXAPERASISDOCIDENNUM = PR.PERASISDOCIDENNUM
	LEFT OUTER JOIN
		CMPER10 PA ON
			B.SOLEXATIPDOCIDENPERCOD = PA.PERTIPDOCIDENCOD 
				AND B.SOLEXAPERASISDOCIDENNUM = PA.PERDOCIDENNUM
	LEFT OUTER JOIN
		CMCAS10 CAS ON
			ACE.ATENAMBCENASICOD = CAS.CENASICOD
	LEFT OUTER JOIN
		CMRAS10 RAS ON
			CAS.REDASISCOD = RAS.REDASISCOD
	LEFT OUTER JOIN
		CMDIA10 DIA ON
			ACE.DIAGCOD = DIA.DIAGCOD
	LEFT OUTER JOIN
		CMAHO10 AHO ON
			B.SOLEXAAREHOSCOD = AHO.AREHOSCOD
	LEFT OUTER JOIN
		CMSHO10 SHO ON
			B.SOLEXASERVHOSCOD = SHO.SERVHOSCOD
	LEFT OUTER JOIN
		CMACT10 ACT ON
			B.SOLEXAACTCOD = ACT.ACTCOD
	LEFT OUTER JOIN
		CMCPP10 CPP ON
			A.RESEXACPSCOD = CPP.CPSCOD
	WHERE 	
		B.SOLEXAFEC >= TO_DATE('{start_date_str}', 'DD/MM/YYYY') AND B.SOLEXAFEC < TO_DATE('{end_date_str}', 'DD/MM/YYYY')
		AND PA.PERESTREGCOD = '1'
		AND PR.PERASISESTREGCOD = '1'
		AND A.RESEXACPSCOD IN ('84478', '82947')
		AND (ACE.DIAGCOD LIKE '%E66%' OR ACE.DIAGCOD LIKE '%E23.6%' OR ACE.DIAGCOD LIKE '%E03.9%' OR ACE.DIAGCOD LIKE '%Z71.3%')
	"""
	print(f'PROCESANDO DE: {start_date_str} A {end_date_str}')
	base = pd.read_sql_query(query, con=connection0)

	# Agrupar por 'solexdfeccita' y 'perdocidennum'
	grouped = base.groupby(['solexdfeccita', 'perdocidennum'])
	new_rows = []
	for name, group in grouped:
		row = {}
		row.update({col: group.iloc[0][col] for col in base.columns[:-6]})
		if len(group) > 1:
			row.update({f"{col}1": group.iloc[0][col] for col in base.columns[-6:]})
			row.update({f"{col}2": group.iloc[1][col] for col in base.columns[-6:]})
		else:
			row.update({f"{col}1": group.iloc[0][col] for col in base.columns[-6:]})
			row.update({f"{col}2": None for col in base.columns[-6:]})
		new_rows.append(row)

	# Crear un nuevo DataFrame con los resultados
	df_combined = pd.DataFrame(new_rows)

	# Guardar los resultados en la base de datos
	df_combined.to_sql(name=f'pnda_labo_cext_obesidad', con=engine1, if_exists='append', index=False)

	# Mover al próximo mes
	current_date = next_month


PROCESANDO DE: 01/01/2020 A 01/02/2020
PROCESANDO DE: 01/02/2020 A 01/03/2020
PROCESANDO DE: 01/03/2020 A 01/04/2020
PROCESANDO DE: 01/04/2020 A 01/05/2020
PROCESANDO DE: 01/05/2020 A 01/06/2020
PROCESANDO DE: 01/06/2020 A 01/07/2020
PROCESANDO DE: 01/07/2020 A 01/08/2020
PROCESANDO DE: 01/08/2020 A 01/09/2020
PROCESANDO DE: 01/09/2020 A 01/10/2020
PROCESANDO DE: 01/10/2020 A 01/11/2020
PROCESANDO DE: 01/11/2020 A 01/12/2020
PROCESANDO DE: 01/12/2020 A 01/01/2021
PROCESANDO DE: 01/01/2021 A 01/02/2021
PROCESANDO DE: 01/02/2021 A 01/03/2021
PROCESANDO DE: 01/03/2021 A 01/04/2021
PROCESANDO DE: 01/04/2021 A 01/05/2021
PROCESANDO DE: 01/05/2021 A 01/06/2021
PROCESANDO DE: 01/06/2021 A 01/07/2021
PROCESANDO DE: 01/07/2021 A 01/08/2021
PROCESANDO DE: 01/08/2021 A 01/09/2021
PROCESANDO DE: 01/09/2021 A 01/10/2021
PROCESANDO DE: 01/10/2021 A 01/11/2021
PROCESANDO DE: 01/11/2021 A 01/12/2021
PROCESANDO DE: 01/12/2021 A 01/01/2022
PROCESANDO DE: 01/01/2022 A 01/02/2022
PROCESANDO DE: 01/02/2022

In [10]:
start_date = datetime(2020, 1, 1)
end_date = datetime(2024, 5, 1)

current_date = start_date

In [11]:
while current_date < end_date:
	next_month = current_date + timedelta(days=31)
	next_month = next_month.replace(day=1)

	# Formatear las fechas para la consulta SQL
	start_date_str = current_date.strftime('%d/%m/%Y')
	end_date_str = next_month.strftime('%d/%m/%Y')

	query = f"""SELECT
		RAS.REDASISCOD,
		RAS.REDASISDES,
		CAS.CENASICOD,
		CAS.CENASIDES,
		DIA.DIAGCOD,
		DIA.DIAGDES,
		ACE.ATENAMBDIAGORD,
		--B.SOLEXAFEC,
		E.SOLEXDFECCITA,
		PA.PERTIPDOCIDENCOD AS TIPDOCMED,
		PA.PERDOCIDENNUM AS NUMDOCMED,
		PA.PERNACFEC AS MEDNACFEC,
		B.SOLEXAAREHOSCOD,
		AHO.AREHOSDES,
		B.SOLEXASERVHOSCOD,
		SHO.SERVHOSDES,
		B.SOLEXAACTCOD,
		ACT.ACTDES,
		G.PERNACFEC,
		G.PERTIPDOCIDENCOD,
		G.PERDOCIDENNUM,
		G.PERSEXOCOD,
		A.RESEXAFEC,
		A.RESEXACPSCOD,
		CPP.CPSDES,
		D.RESEXDEXADES,
		D.RESEXDEXAUND,
		D.RESEXDEXANORVAL
	FROM CTDAA10 ACE
	LEFT OUTER JOIN 
		ETSEA10 B ON 
			ACE.ATENAMBORICENASICOD  = B.SOLEXAORICENASIORICOD
				AND ACE.ATENAMBCENASICOD  = B.SOLEXACENASIORICOD
				AND ACE.ATENAMBNUM  = B.SOLEXAACTMEDORINUM
	LEFT OUTER JOIN
		ETSED10 E ON
			E.SOLEXAORICENASICOD = B.SOLEXAORICENASICOD
				AND E.SOLEXACENASICOD = B.SOLEXACENASICOD
				AND E.SOLEXATIPEXACOD = B.SOLEXATIPEXACOD 
				AND E.SOLEXANUM = B.SOLEXANUM
	LEFT OUTER JOIN 
		ETREA10 A ON 
			E.SOLEXAORICENASICOD = A.RESEXAORICENASICOD
				AND E.SOLEXACENASICOD = A.RESEXACENASICOD
				AND E.SOLEXATIPEXACOD = A.RESEXATIPEXACOD
				AND E.SOLEXANUM = A.RESEXASOLEXANUM
				AND E.SOLEXDCPSCOD = A.RESEXACPSCOD
	LEFT OUTER JOIN 
		ETRED10 D ON 
			D.RESEXAORICENASICOD = A.RESEXAORICENASICOD
				AND D.RESEXACENASICOD = A.RESEXACENASICOD
				AND D.RESEXATIPEXACOD = A.RESEXATIPEXACOD
				AND D.RESEXASOLEXANUM = A.RESEXASOLEXANUM
				AND D.RESEXACPSCOD = A.RESEXACPSCOD
	LEFT OUTER JOIN 
		CMPER10 G ON 
			G.PERSECNUM = B.SOLEXAPACSECNUM
	LEFT OUTER JOIN
		CMPRS10 PR ON
			B.SOLEXATIPDOCIDENPERCOD = PR.TIPDOCIDENPERCOD 
				AND B.SOLEXAPERASISDOCIDENNUM = PR.PERASISDOCIDENNUM
	LEFT OUTER JOIN
		CMPER10 PA ON
			B.SOLEXATIPDOCIDENPERCOD = PA.PERTIPDOCIDENCOD 
				AND B.SOLEXAPERASISDOCIDENNUM = PA.PERDOCIDENNUM
	LEFT OUTER JOIN
		CMCAS10 CAS ON
			ACE.ATENAMBCENASICOD = CAS.CENASICOD
	LEFT OUTER JOIN
		CMRAS10 RAS ON
			CAS.REDASISCOD = RAS.REDASISCOD
	LEFT OUTER JOIN
		CMDIA10 DIA ON
			ACE.DIAGCOD = DIA.DIAGCOD
	LEFT OUTER JOIN
		CMAHO10 AHO ON
			B.SOLEXAAREHOSCOD = AHO.AREHOSCOD
	LEFT OUTER JOIN
		CMSHO10 SHO ON
			B.SOLEXASERVHOSCOD = SHO.SERVHOSCOD
	LEFT OUTER JOIN
		CMACT10 ACT ON
			B.SOLEXAACTCOD = ACT.ACTCOD
	LEFT OUTER JOIN
		CMCPP10 CPP ON
			A.RESEXACPSCOD = CPP.CPSCOD
	WHERE 	
		B.SOLEXAFEC >= TO_DATE('{start_date_str}', 'DD/MM/YYYY') AND B.SOLEXAFEC < TO_DATE('{end_date_str}', 'DD/MM/YYYY')
		AND PA.PERESTREGCOD = '1'
		AND PR.PERASISESTREGCOD = '1'
		AND A.RESEXACPSCOD IN ('84478', '82565')
		AND (ACE.DIAGCOD LIKE '%I10%' OR ACE.DIAGCOD LIKE '%I11%' OR ACE.DIAGCOD LIKE '%I12%' OR ACE.DIAGCOD LIKE '%I15%' OR ACE.DIAGCOD LIKE '%O10%')
	"""
	print(f'PROCESANDO DE: {start_date_str} A {end_date_str}')
	base = pd.read_sql_query(query, con=connection0)

	# Agrupar por 'solexdfeccita' y 'perdocidennum'
	grouped = base.groupby(['solexdfeccita', 'perdocidennum'])
	new_rows = []
	for name, group in grouped:
		row = {}
		row.update({col: group.iloc[0][col] for col in base.columns[:-6]})
		if len(group) > 1:
			row.update({f"{col}1": group.iloc[0][col] for col in base.columns[-6:]})
			row.update({f"{col}2": group.iloc[1][col] for col in base.columns[-6:]})
		else:
			row.update({f"{col}1": group.iloc[0][col] for col in base.columns[-6:]})
			row.update({f"{col}2": None for col in base.columns[-6:]})
		new_rows.append(row)

	# Crear un nuevo DataFrame con los resultados
	df_combined = pd.DataFrame(new_rows)

	# Guardar los resultados en la base de datos
	df_combined.to_sql(name=f'pnda_labo_cext_hipertension', con=engine1, if_exists='append', index=False)

	# Mover al próximo mes
	current_date = next_month

PROCESANDO DE: 01/01/2020 A 01/02/2020
PROCESANDO DE: 01/02/2020 A 01/03/2020
PROCESANDO DE: 01/03/2020 A 01/04/2020
PROCESANDO DE: 01/04/2020 A 01/05/2020
PROCESANDO DE: 01/05/2020 A 01/06/2020
PROCESANDO DE: 01/06/2020 A 01/07/2020
PROCESANDO DE: 01/07/2020 A 01/08/2020
PROCESANDO DE: 01/08/2020 A 01/09/2020
PROCESANDO DE: 01/09/2020 A 01/10/2020
PROCESANDO DE: 01/10/2020 A 01/11/2020
PROCESANDO DE: 01/11/2020 A 01/12/2020
PROCESANDO DE: 01/12/2020 A 01/01/2021
PROCESANDO DE: 01/01/2021 A 01/02/2021
PROCESANDO DE: 01/02/2021 A 01/03/2021
PROCESANDO DE: 01/03/2021 A 01/04/2021
PROCESANDO DE: 01/04/2021 A 01/05/2021
PROCESANDO DE: 01/05/2021 A 01/06/2021
PROCESANDO DE: 01/06/2021 A 01/07/2021
PROCESANDO DE: 01/07/2021 A 01/08/2021
PROCESANDO DE: 01/08/2021 A 01/09/2021
PROCESANDO DE: 01/09/2021 A 01/10/2021
PROCESANDO DE: 01/10/2021 A 01/11/2021
PROCESANDO DE: 01/11/2021 A 01/12/2021
PROCESANDO DE: 01/12/2021 A 01/01/2022
PROCESANDO DE: 01/01/2022 A 01/02/2022
PROCESANDO DE: 01/02/2022

In [4]:
start_date = datetime(2020, 1, 1)
end_date = datetime(2024, 5, 1)

current_date = start_date

In [5]:
while current_date < end_date:
	next_month = current_date + timedelta(days=31)
	next_month = next_month.replace(day=1)

	# Formatear las fechas para la consulta SQL
	start_date_str = current_date.strftime('%d/%m/%Y')
	end_date_str = next_month.strftime('%d/%m/%Y')

	query = f"""SELECT
		RAS.REDASISCOD,
		RAS.REDASISDES,
		CAS.CENASICOD,
		CAS.CENASIDES,
		DIA.DIAGCOD,
		DIA.DIAGDES,
		ACE.ATENAMBDIAGORD,
		--B.SOLEXAFEC,
		E.SOLEXDFECCITA,
		PA.PERTIPDOCIDENCOD AS TIPDOCMED,
		PA.PERDOCIDENNUM AS NUMDOCMED,
		PA.PERNACFEC AS MEDNACFEC,
		B.SOLEXAAREHOSCOD,
		AHO.AREHOSDES,
		B.SOLEXASERVHOSCOD,
		SHO.SERVHOSDES,
		B.SOLEXAACTCOD,
		ACT.ACTDES,
		G.PERNACFEC,
		G.PERTIPDOCIDENCOD,
		G.PERDOCIDENNUM,
		G.PERSEXOCOD,
		A.RESEXAFEC,
		A.RESEXACPSCOD,
		CPP.CPSDES,
		D.RESEXDEXADES,
		D.RESEXDEXAUND,
		D.RESEXDEXANORVAL
	FROM CTDAA10 ACE
	LEFT OUTER JOIN 
		ETSEA10 B ON 
			ACE.ATENAMBORICENASICOD  = B.SOLEXAORICENASIORICOD
				AND ACE.ATENAMBCENASICOD  = B.SOLEXACENASIORICOD
				AND ACE.ATENAMBNUM  = B.SOLEXAACTMEDORINUM
	LEFT OUTER JOIN
		ETSED10 E ON
			E.SOLEXAORICENASICOD = B.SOLEXAORICENASICOD
				AND E.SOLEXACENASICOD = B.SOLEXACENASICOD
				AND E.SOLEXATIPEXACOD = B.SOLEXATIPEXACOD 
				AND E.SOLEXANUM = B.SOLEXANUM
	LEFT OUTER JOIN 
		ETREA10 A ON 
			E.SOLEXAORICENASICOD = A.RESEXAORICENASICOD
				AND E.SOLEXACENASICOD = A.RESEXACENASICOD
				AND E.SOLEXATIPEXACOD = A.RESEXATIPEXACOD
				AND E.SOLEXANUM = A.RESEXASOLEXANUM
				AND E.SOLEXDCPSCOD = A.RESEXACPSCOD
	LEFT OUTER JOIN 
		ETRED10 D ON 
			D.RESEXAORICENASICOD = A.RESEXAORICENASICOD
				AND D.RESEXACENASICOD = A.RESEXACENASICOD
				AND D.RESEXATIPEXACOD = A.RESEXATIPEXACOD
				AND D.RESEXASOLEXANUM = A.RESEXASOLEXANUM
				AND D.RESEXACPSCOD = A.RESEXACPSCOD
	LEFT OUTER JOIN 
		CMPER10 G ON 
			G.PERSECNUM = B.SOLEXAPACSECNUM
	LEFT OUTER JOIN
		CMPRS10 PR ON
			B.SOLEXATIPDOCIDENPERCOD = PR.TIPDOCIDENPERCOD 
				AND B.SOLEXAPERASISDOCIDENNUM = PR.PERASISDOCIDENNUM
	LEFT OUTER JOIN
		CMPER10 PA ON
			B.SOLEXATIPDOCIDENPERCOD = PA.PERTIPDOCIDENCOD 
				AND B.SOLEXAPERASISDOCIDENNUM = PA.PERDOCIDENNUM
	LEFT OUTER JOIN
		CMCAS10 CAS ON
			ACE.ATENAMBCENASICOD = CAS.CENASICOD
	LEFT OUTER JOIN
		CMRAS10 RAS ON
			CAS.REDASISCOD = RAS.REDASISCOD
	LEFT OUTER JOIN
		CMDIA10 DIA ON
			ACE.DIAGCOD = DIA.DIAGCOD
	LEFT OUTER JOIN
		CMAHO10 AHO ON
			B.SOLEXAAREHOSCOD = AHO.AREHOSCOD
	LEFT OUTER JOIN
		CMSHO10 SHO ON
			B.SOLEXASERVHOSCOD = SHO.SERVHOSCOD
	LEFT OUTER JOIN
		CMACT10 ACT ON
			B.SOLEXAACTCOD = ACT.ACTCOD
	LEFT OUTER JOIN
		CMCPP10 CPP ON
			A.RESEXACPSCOD = CPP.CPSCOD
	WHERE 	
		B.SOLEXAFEC >= TO_DATE('{start_date_str}', 'DD/MM/YYYY') AND B.SOLEXAFEC < TO_DATE('{end_date_str}', 'DD/MM/YYYY')
		AND PA.PERESTREGCOD = '1'
		AND PR.PERASISESTREGCOD = '1'
		AND A.RESEXACPSCOD IN ('84478', '82565')
		AND ACE.DIAGCOD LIKE '%E78%'
	"""
	print(f'PROCESANDO DE: {start_date_str} A {end_date_str}')
	base = pd.read_sql_query(query, con=connection0)

	# Agrupar por 'solexdfeccita' y 'perdocidennum'
	grouped = base.groupby(['solexdfeccita', 'perdocidennum'])
	new_rows = []
	for name, group in grouped:
		row = {}
		row.update({col: group.iloc[0][col] for col in base.columns[:-6]})
		if len(group) > 1:
			row.update({f"{col}1": group.iloc[0][col] for col in base.columns[-6:]})
			row.update({f"{col}2": group.iloc[1][col] for col in base.columns[-6:]})
		else:
			row.update({f"{col}1": group.iloc[0][col] for col in base.columns[-6:]})
			row.update({f"{col}2": None for col in base.columns[-6:]})
		new_rows.append(row)

	# Crear un nuevo DataFrame con los resultados
	df_combined = pd.DataFrame(new_rows)

	# Guardar los resultados en la base de datos
	df_combined.to_sql(name=f'pnda_labo_cext_hiperlipidemia', con=engine1, if_exists='append', index=False)

	# Mover al próximo mes
	current_date = next_month

PROCESANDO DE: 01/01/2020 A 01/02/2020
PROCESANDO DE: 01/02/2020 A 01/03/2020
PROCESANDO DE: 01/03/2020 A 01/04/2020
PROCESANDO DE: 01/04/2020 A 01/05/2020
PROCESANDO DE: 01/05/2020 A 01/06/2020
PROCESANDO DE: 01/06/2020 A 01/07/2020
PROCESANDO DE: 01/07/2020 A 01/08/2020
PROCESANDO DE: 01/08/2020 A 01/09/2020
PROCESANDO DE: 01/09/2020 A 01/10/2020
PROCESANDO DE: 01/10/2020 A 01/11/2020
PROCESANDO DE: 01/11/2020 A 01/12/2020
PROCESANDO DE: 01/12/2020 A 01/01/2021
PROCESANDO DE: 01/01/2021 A 01/02/2021
PROCESANDO DE: 01/02/2021 A 01/03/2021
PROCESANDO DE: 01/03/2021 A 01/04/2021
PROCESANDO DE: 01/04/2021 A 01/05/2021
PROCESANDO DE: 01/05/2021 A 01/06/2021
PROCESANDO DE: 01/06/2021 A 01/07/2021
PROCESANDO DE: 01/07/2021 A 01/08/2021
PROCESANDO DE: 01/08/2021 A 01/09/2021
PROCESANDO DE: 01/09/2021 A 01/10/2021
PROCESANDO DE: 01/10/2021 A 01/11/2021
PROCESANDO DE: 01/11/2021 A 01/12/2021
PROCESANDO DE: 01/12/2021 A 01/01/2022
PROCESANDO DE: 01/01/2022 A 01/02/2022
PROCESANDO DE: 01/02/2022

In [ ]:
connection0.close()
engine0.dispose()
connection1.close()
engine1.dispose()